Accelerator: GPU T4 x2, Internet on, and attach the `kaggle_prepare_data` output under Add Input → Your Work → Notebook Output.

In [1]:
REPO_URL = "https://github.com/Splestule/candidate_reranker.git"
BRANCH = "main"

In [2]:
import subprocess, sys
from pathlib import Path

CODE = Path("/kaggle/working/candidate_reranker")
if not (CODE / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(CODE)],
                   check=True)

sys.path.insert(0, str(CODE / "src"))
import kaggle_env as K

COMMIT = K.sync(REPO_URL, BRANCH)     # rerun this cell after every push
K.gpu_info()
env = K.prepare(COMMIT)

Cloning into '/kaggle/working/candidate_reranker'...
From https://github.com/Splestule/candidate_reranker
 * branch            main       -> FETCH_HEAD


HEAD is now at 7262276 Merge remote-tracking branch 'origin/main'
torch 2.10.0+cu128  cuda=True
Tesla T4  compute capability 7.5
Turing/Pascal: no bf16, no FlashAttention 2 -- handled by wf_compat
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 586.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 25.4 MB/s eta 0:00:00


Cloning into '/kaggle/working/Whisfusion'...


commit      7262276
base model  /kaggle/input/notebooks/eduardimon/prepare-data-for-word-reranker/ckpt/mdm_safetensors/mdm-170M-100e18-rsl-0.01.safetensors
adapter     /kaggle/input/notebooks/eduardimon/prepare-data-for-word-reranker/ckpt/whisfusion_stage2_decoder.pt
librispeech /kaggle/input/notebooks/eduardimon/prepare-data-for-word-reranker/data/LibriSpeech  ['dev-clean', 'test-clean', 'test-other']
hf cache    /kaggle/working/hf  prepared
ood         /kaggle/input/notebooks/eduardimon/prepare-data-for-word-reranker/ood  ['irish_english_male', 'midlands_english_female', 'northern_english_female']


In [3]:
rc = K.run(env, "selftest.py",
           "--base_model", env.base_model,
           "--adapter", env.adapter,
           "--librispeech", env.librispeech / "test-clean",
           "--n_utts", "3",
           "--n_candidates", "5")
assert rc == 0, f"selftest exit code {rc}, see the output above"

[wf_compat] shims active: rotary_emb, dropout_layer_norm, flash_attn, xformers.ops.SwiGLU, FusedRMSNorm
SELFTEST
torch 2.10.0+cu128  cuda=True
GPU: Tesla T4  compute capability 7.5
     Turing/Pascal: no bf16, no FlashAttention 2 -> fp16 + SDPA
[1/5] compat shims OK
[wf] device=cuda (Tesla T4) dtype=torch.float16
[wf] base: 129 keys, 72 missing
[wf] adapter: 201 keys, 0 missing, 0 unexpected

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 1951.09it/s, Materializing param=model.encoder.layers.11.self_attn_layer_norm.weight]
[wf] decoder 261.5M · Whisper encoder 88.2M
[2/5] checkpoint loaded with no missing keys
  1089-134686-0000  10.4s  WER  10.7 (oracle  10.7, 5/5 unique)
      REF he hoped there would be stew for dinner turnips and carrots and brui
      HYP he hoped there would be stew for dinner turnips and carrots and brui
  1089-134686-0001   3.3s  WER  25.0 (oracle  25.0, 4/5 unique)
      REF stuff it into you his belly counselled him
      HYP stuffed it into you his 

Manifest for the dialect audio. Paths differ between notebooks, so it is built here.

In [4]:
DIALECTS = env.results / "dialects.jsonl"
K.run(env, "make_manifest.py",
      "--slr83", *sorted(p for p in env.ood.iterdir() if p.is_dir()),
      "--out", DIALECTS)

irish_english_male: 450 utterances
midlands_english_female: 246 utterances
northern_english_female: 750 utterances

1446 utterances, 153.3 min of audio -> /kaggle/working/results/dialects.jsonl


0

Dump candidates. Roughly 2 hours of GPU for all four.

In [5]:
def dump(tag, source, path, **kw):
    args = ["--source", source, "--path", path,
            "--base_model", env.base_model, "--adapter", env.adapter,
            "--out", env.results / f"{tag}-{COMMIT}.jsonl",
            "--tag", tag, "--resume"]
    for k, v in kw.items():
        args += [f"--{k}", v]
    return K.run(env, "dump_candidates.py", *args)

dump("test-clean", "librispeech", env.librispeech / "test-clean")
dump("test-other", "librispeech", env.librispeech / "test-other")
dump("dialects", "manifest", DIALECTS)
dump("dev-clean", "librispeech", env.librispeech / "dev-clean", max_utts=800)

[wf_compat] shims active: rotary_emb, dropout_layer_norm, flash_attn, xformers.ops.SwiGLU, FusedRMSNorm
[wf] device=cuda (Tesla T4) dtype=torch.float16
[wf] base: 129 keys, 72 missing
[wf] adapter: 201 keys, 0 missing, 0 unexpected

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 1978.12it/s, Materializing param=model.encoder.layers.11.self_attn_layer_norm.weight]
[wf] decoder 261.5M · Whisper encoder 88.2M
  25 utts · 0.4 min · 0.94 s/utt · RTF 0.109
  50 utts · 0.8 min · 0.94 s/utt · RTF 0.125
  75 utts · 1.2 min · 0.95 s/utt · RTF 0.116
  100 utts · 1.6 min · 0.98 s/utt · RTF 0.109
  125 utts · 2.1 min · 1.01 s/utt · RTF 0.117
  150 utts · 2.5 min · 1.01 s/utt · RTF 0.117
  175 utts · 3.0 min · 1.02 s/utt · RTF 0.119
  200 utts · 3.4 min · 1.03 s/utt · RTF 0.113
  225 utts · 3.9 min · 1.03 s/utt · RTF 0.115
  250 utts · 4.3 min · 1.04 s/utt · RTF 0.118
  275 utts · 4.8 min · 1.04 s/utt · RTF 0.118
  300 utts · 5.2 min · 1.04 s/utt · RTF 0.118
  325 utts · 5.7 min · 1.04 s/ut

0

Tune on dev, then measure on all three test sets.

In [6]:
K.run(env, "tune.py",
      "--dev", env.results / f"dev-clean-{COMMIT}.jsonl",
      "--test", env.results / f"test-clean-{COMMIT}.jsonl",
      "--json", env.results / f"tune-{COMMIT}.json")

dev: 800 utterances from 1 dump(s)

lambda for conf + lambda*mbr
   0.0    7.93
   0.1    7.55
  0.25    7.29
   0.5    7.20
  0.75    7.11
   1.0    7.09
   1.5    7.08
   2.0    7.20
   3.0    7.22

ROVER alpha (1.0 = votes only) and epsilon confidence
  alpha 0.3   eps 0.3    13.86
  alpha 0.3   eps 0.5    10.59
  alpha 0.3   eps 0.7     5.75
  alpha 0.5   eps 0.3     6.18
  alpha 0.5   eps 0.5     5.77
  alpha 0.5   eps 0.7     5.69
  alpha 0.7   eps 0.3     5.84
  alpha 0.7   eps 0.5     5.80
  alpha 0.7   eps 0.7     5.78
  alpha 0.85  eps 0.3     5.84
  alpha 0.85  eps 0.5     5.83
  alpha 0.85  eps 0.7     5.80
  alpha 1.0   eps 0.3     5.92
  alpha 1.0   eps 0.5     5.92
  alpha 1.0   eps 0.7     5.92

chosen on dev: lambda=1.5  alpha=0.5  eps_conf=0.7

test: 2620 utterances from /kaggle/working/results/test-clean-7262276.jsonl
  pick by confidence        9.04
  pick with tuned lambda    8.32
  ROVER with tuned params   6.53

written: /kaggle/working/results/tune-7262276.json


0

In [7]:
import json

tuned = json.load(open(env.results / f"tune-{COMMIT}.json"))

for tag in ["test-clean", "test-other", "dialects"]:
    print("\n" + "#" * 74 + f"\n# {tag}\n" + "#" * 74)
    K.run(env, "analyze.py", env.results / f"{tag}-{COMMIT}.jsonl",
          "--json", env.results / f"{tag}-{COMMIT}.stats.json")
    K.run(env, "analyze_compose.py", env.results / f"{tag}-{COMMIT}.jsonl",
          "--alpha", tuned["alpha"], "--eps_conf", tuned["eps_conf"],
          "--json", env.results / f"compose-{tag}-{COMMIT}.json")


##########################################################################
# test-clean
##########################################################################
ORACLE GAP  ·  /kaggle/working/results/test-clean-7262276.jsonl
utterances: 2620   K: 15   audio: 324.2 min
identical candidates after step 1: 100.0 % of utts

--------------------------------------------------------------------------
scorer                        corpus WER   mean-utt     hit %       gap
--------------------------------------------------------------------------
mean_conf (upstream)                9.04       8.24      69.9      2.43
min_conf                           10.50       9.26      62.4      3.46
median_conf                        10.37       9.49      57.9      3.69
mean_logprob                        9.11       8.28      70.1      2.48
neg_entropy                         9.06       8.23      70.3      2.43
len_norm_conf                       9.19       8.67      67.2      2.87
mbr_wer               